# TP2 - Ejercicios 2 y 3: Aprendizaje por refuerzo

Q-Learning tabular y DQN con NumPy, Gymnasium y PyTorch.

## 1. Taxi-v3: definición del MDP; Navegación urbana

Modelamos la navegación autónoma del agente a través del tráfico cruzado. 

* El estado codifica fila, columna, posición del pasajero y destino (25×5×4 = 500 estados). 

* Las seis acciones son cuatro movimientos cardinales, subir y bajar al cliente. 

* El modelo de transición es determinista dado estado y acción con los valores por defecto y los estados iniciales se sortean. 

* La recompensa base es −1 por paso, −10 por recogida o descenso ilegal y +20 por entrega correcta. γ=0,97 balancea la urgencia de entrega con la exploración.

In [1]:
import os
import time
import math
import json
import random
from collections import deque
from pathlib import Path
from IPython.display import Image, Video, display
import gymnasium as gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn

ROOT = Path.cwd()
PROFILE = os.getenv("TP2_PROFILE", "full")
SMOKE = PROFILE == "smoke"
OUT = ROOT / "resultados" / PROFILE
OUT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(min(2, os.cpu_count() or 1))
MAX_DQN_EPISODES = 700

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def save_json(name, result):
    (OUT / (name + ".json")).write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")

def save_checkpoint(path, obj):
    temporary = path.with_suffix(".tmp")
    torch.save(obj, temporary)
    temporary.replace(path)

def rolling(values, window):
    values = np.asarray(values, dtype=float)
    cumsum = np.cumsum(np.insert(values, 0, 0))
    return (cumsum[window:] - cumsum[:-window]) / window

def first_cross(history, threshold, window=50):
    if len(history) < window:
        return None
    means = rolling(history, window)
    hits = np.flatnonzero(means >= threshold)
    return int(hits[0] + window) if len(hits) else None

def smooth(values, weight=0.9):
    smoothed, running = [], 0.0
    for value in values:
        running = value if not smoothed else weight * running + (1 - weight) * value
        smoothed.append(running)
    return smoothed

def save_figure(fig, name, show=False):
    fig.tight_layout()
    fig.savefig(OUT / f"{name}.png", dpi=145)
    if show:
        plt.show()
    plt.close(fig)

def obtener_resultado(nombre, ejecutar):
    ruta = OUT / (nombre + ".json")
    return json.loads(ruta.read_text(encoding="utf-8")) if ruta.is_file() else ejecutar()

seed_everything(42)
taxi_env = gym.make("Taxi-v3")
print(taxi_env.observation_space, taxi_env.action_space)
print("Entradas:", taxi_env.observation_space.n * taxi_env.action_space.n)
taxi_env.close()


Discrete(500) Discrete(6)
Entradas: 3000


La tabla contiene 3.000 posiciones posibles, aunque no todas las combinaciones son alcanzables en la misma fase del episodio.